In [1]:
import os
from openai import OpenAI

token = os.environ["GITHUB_TOKEN"]
endpoint = "https://models.github.ai/inference"
# Cambiamos a un modelo que SÍ permite ajustar temperatura
model_name = "gpt-4o-mini" 

client = OpenAI(
    base_url=endpoint,
    api_key=token,
)

response = client.chat.completions.create(
    messages=[
        {
            "role": "system", # Usamos system para mayor compatibilidad
            "content": "Eres mi asistente pero muy huaso.",
        },
        {
            "role": "user",
            "content": "cuentame una historia",
        }
    ],
    model=model_name,
    temperature=0.2  # <--- AQUÍ AJUSTAS LA TEMPERATURA (0.0 a 2.0)
)

print(response.choices[0].message.content)

Había una vez en un pequeño pueblo llamado Valle Verde, un joven llamado Tomás que soñaba con ser aventurero. Cada día, mientras ayudaba a su padre en la granja, miraba hacia las montañas que rodeaban el pueblo y se imaginaba explorando sus misterios.

Un día, mientras paseaba por el bosque cercano, Tomás encontró un viejo mapa escondido entre las raíces de un árbol. El mapa mostraba un camino hacia un tesoro escondido en la cima de la montaña más alta. Emocionado, decidió que era el momento de embarcarse en su propia aventura.

Tomás se preparó con lo que tenía: una mochila, un poco de comida y su fiel perro, Rocco. Juntos, comenzaron la travesía. El camino era difícil y lleno de obstáculos, pero Tomás no se rindió. Con cada paso, aprendía a superar sus miedos y a confiar en sí mismo.

Después de días de caminata, finalmente llegaron a la cima de la montaña. Allí, encontraron un cofre antiguo cubierto de polvo. Con el corazón latiendo de emoción, Tomás lo abrió y, para su sorpresa, no

In [2]:
import os
import logging
from pathlib import Path

from dotenv import load_dotenv, find_dotenv

_dotenv_path = find_dotenv(filename=".env", usecwd=True)
if _dotenv_path:
    load_dotenv(_dotenv_path)
else:
    for _dir in (Path.cwd(), *list(Path.cwd().parents)[:8]):
        _f = _dir / ".env"
        if _f.is_file():
            load_dotenv(_f)
            break
    else:
        load_dotenv()

_ls_key = os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGCHAIN_API_KEY")
if _ls_key:
    os.environ.setdefault("LANGCHAIN_API_KEY", _ls_key)
    os.environ.setdefault("LANGSMITH_API_KEY", _ls_key)

if os.getenv("LANGSMITH_TRACING", "").lower() == "true":
    os.environ["LANGCHAIN_TRACING_V2"] = "true"

import langsmith

if os.getenv("LANGSMITH_TRACING", "").lower() == "true" and _ls_key:
    langsmith.configure(
        enabled=True,
        project_name=os.getenv("LANGSMITH_PROJECT") or "default",
    )

if os.getenv("LANGCHAIN_DEBUG", "").lower() == "true":
    from langchain_core.globals import set_debug

    set_debug(True)

if os.getenv("LANGSMITH_TRACING", "").lower() == "true":
    print(
        "[LangSmith]",
        "proyecto=" + os.getenv("LANGSMITH_PROJECT", "(sin LANGSMITH_PROJECT)"),
        "| api_key=" + ("ok" if _ls_key else "FALTA — sin esto no hay trazas"),
        "| .env=" + (_dotenv_path or "buscado desde cwd"),
    )

# Menos ruido en consola solo si no usas LangSmith
if os.getenv("LANGSMITH_TRACING", "").lower() != "true":
    logging.getLogger("langsmith").setLevel(logging.ERROR)
    logging.getLogger("langchain").setLevel(logging.ERROR)

# 3. (Opcional) Si quieres ser aún más radical con los mensajes de Pydantic
import warnings
warnings.filterwarnings("ignore")


# ... el resto de tu código
from langchain_openai import ChatOpenAI
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

token = os.environ.get("GITHUB_TOKEN")
# URL base estándar para GitHub Models
endpoint = "https://models.github.ai/inference" 
model_name = "gpt-4o-mini"

llm = ChatOpenAI(
    base_url=endpoint,
    api_key=token,
    model=model_name,
    temperature=0.3,
    # Desactivamos streaming si no lo estamos manejando con .stream() 
    # para evitar errores de protocolo HTTP en algunos modelos
    streaming=False 
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres 'Manantial', el asistente virtual de la empresa Manantial. Eres amable y servicial."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

chain = prompt | llm
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

def chat(mensaje, s_id="sesion_1"):
    try:
        response = conversation.invoke(
            {"input": mensaje},
            config={"configurable": {"session_id": s_id}}
        )
        print(f"🔹 Cliente: {mensaje}")
        print(f"💧 Manantial: {response.content}")
        print("-" * 50)
    except Exception as e:
        print(f"❌ Error al conectar con Manantial: {e}")

# Ejecución
chat("¡Hola! Me llamo Jose.")
chat("¿Cuál es mi nombre?")

try:
    from langchain_core.tracers.langchain import wait_for_all_tracers

    wait_for_all_tracers()
except Exception:
    pass

🔹 Cliente: ¡Hola! Me llamo Jose.
💧 Manantial: ¡Hola, Jose! Es un placer conocerte. ¿En qué puedo ayudarte hoy?
--------------------------------------------------
🔹 Cliente: ¿Cuál es mi nombre?
💧 Manantial: Tu nombre es Jose. ¿Hay algo más en lo que pueda ayudarte?
--------------------------------------------------
